# PA1 Task 4 (Open-Set Recognition) on Kaggle

1. Upload `pa1_task4_kaggle.zip` as a Kaggle **Dataset** (any name), then in this notebook: **Add Input** → that dataset.
2. Notebook settings: **Accelerator = GPU T4 x2**, **Internet = On**.
3. Run the cells top to bottom. If a training cell stops, run it again: each run resumes from its last finished epoch.

In [ ]:
# Copy the NEWEST uploaded code into /kaggle/working/pa1 (overwrites code files, keeps results/checkpoints).
import glob, os, shutil
cands = [os.path.dirname(os.path.dirname(os.path.dirname(p))) for p in glob.glob("/kaggle/input/**/task4/data/cifar.py", recursive=True)]
new = [c for c in cands if "huggingface" in open(f"{c}/task4/data/cifar.py").read()]
print("found:", cands)
assert new, "Only the OLD code is attached: open the Input panel and update/add the new dataset version."
shutil.copytree(new[0], "/kaggle/working/pa1", dirs_exist_ok=True)
%cd /kaggle/working/pa1
os.environ["PA1_DATA_ROOT"] = "/kaggle/working/data"
os.environ["PA1_OUT_ROOT"] = "/kaggle/working"
os.environ["PYTHONUNBUFFERED"] = "1"  # logs show each epoch as soon as it finishes
os.makedirs("logs", exist_ok=True)
!grep -c huggingface task4/data/cifar.py

In [ ]:
# 1. Preflight (~3 min): every Task 4 step for a few batches on this GPU, in a temp folder. Also downloads CIFAR.
# Do NOT continue unless the last line says PREFLIGHT OK.
!bash task4/preflight.sh

In [ ]:
# 2. Vanilla and GCSC at the same time, one per GPU. Prints the latest epoch of each every minute.
# Full logs: logs/vanilla.log, logs/gcsc.log. If this cell stops, run it again: runs resume from their last epoch.
import subprocess, time

def run_and_watch(cmd, logs, every=60):
    """Run cmd in the background and print each log's latest line every `every` seconds."""
    p = subprocess.Popen(cmd, shell=True)
    while p.poll() is None:
        time.sleep(every)
        for n in logs:
            try:
                lines = open(f"logs/{n}.log").read().splitlines()
                print(n, "|", lines[-1][:110] if lines else "(starting)")
            except FileNotFoundError:
                print(n, "| (starting)")
    print("finished, exit code", p.returncode)

run_and_watch("bash kaggle/run_parallel.sh task4.train task4/configs/vanilla.yaml task4/configs/gcsc.yaml "
              "train.num_workers=2", ["vanilla", "gcsc"])

In [ ]:
# 3. PROSER, starting from the vanilla checkpoint (50 epochs). Log: logs/proser.log
run_and_watch("python -m task4.train --config task4/configs/proser.yaml train.num_workers=2 > logs/proser.log 2>&1",
              ["proser"])

In [ ]:
# 3b. PROSER's final-epoch model (extra rows in the table, NOT the main PROSER result). Uses no unknown data.
import torch
from common.io import checkpoint_dir
last = torch.load(checkpoint_dir("task4", "proser") / "last.pt", map_location="cpu", weights_only=False)
torch.save({"model": last["model"], "epoch": last["epoch"], "val_accuracy": None,
            "num_outputs": last["model"]["net.fc.weight"].shape[0], "config": last["config"],
            "note": "PROSER after its final (50th) epoch, NOT selected by validation accuracy"},
           checkpoint_dir("task4", "proser_final") / "best.pt")
print("proser_final = epoch", last["epoch"])

In [ ]:
# 4. ONLY after all three models are trained: save outputs (incl. CIFAR-100 unknowns), then evaluate.
!for r in vanilla gcsc proser proser_final; do python -m task4.extract_outputs --run $r --include-unknowns; done
!python -m task4.evaluate_osr

In [ ]:
# 5. Pack everything to download (Output tab, or the file browser on the right).
!cd /kaggle/working && zip -qr task4_out.zip pa1/task4/results pa1/task4/cache pa1/logs checkpoints/task4 && ls -lh task4_out.zip